In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import os
import glob
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
df = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)], ignore_index=True)

print(f"✅ Dataset berhasil digabung! Total: {len(df)} baris")
print(df.head())

✅ Dataset berhasil digabung! Total: 15641 baris
                  Title                                        Ingredients  \
0         .Sate Kambing  Bahan-bahan :--500 gr Daging kambing--Daun pep...   
1         Rabeg Kambing  1 kg daging kambing bagian paha beserta tulang...   
2         Gulai kambing  500 gram daging kambing--Bumbu :--sesuai seler...   
3  Sayur tulang kambing  1.5 kg tulang sumsum kambing(direbus dahulu)ag...   
4          Sate Kambing  300 garam daging kambing--1/2 buah jeruk nipis...   

                                               Steps  Loves  \
0  1. Cuci bersih daging kambing, potong" kotak, ...      6   
1  Tumis bumbu halus hingga harum. Masukan bumbu ...      6   
2  Rebus dgng kambing dgn jahe krng lbh 20 menit ...      0   
3  Haluskan bawang merah 5 siung dan 5 bawang put...      1   
4  Potong daging kambing sesuai selera, beri pera...      7   

                                      URL  
0          /id/resep/4470066-sate-kambing  
1         /id/re

In [13]:
stopwords_dapur = [
    'gram', 'kg', 'ml', 'liter', 'siung', 'lembar', 'sdt', 'sdm', 'sendok', 
    'makan', 'teh', 'secukupnya', 'ruas', 'ikat', 'potong', 'dadu', 'memarkan', 
    'iris', 'tipis', 'bungkus', 'biji', 'buah', 'helai', 'batang', 'geprek', 'haluskan'
]

def clean_ingredients(text):
    if pd.isna(text): return ""
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    return " ".join([w for w in text.split() if w not in stopwords_dapur and len(w) > 2])

df['Cleaned_Ingredients'] = df['Ingredients'].apply(clean_ingredients)

In [14]:
vectorizer = CountVectorizer(max_features=500, binary=True)
X_ingredients = vectorizer.fit_transform(df['Cleaned_Ingredients']).toarray()

X_porsi = np.random.randint(10, 100, size=(len(df), 1)).astype(float)
X = np.hstack((X_ingredients, X_porsi))

In [15]:
jumlah_bahan_per_resep = np.sum(X_ingredients, axis=1)
y_hpp = (jumlah_bahan_per_resep * 2000) * X_porsi[:, 0] + 50000

In [16]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_dim=X.shape[1], activation='linear')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=50.0), loss='mae')
model.fit(X, y_hpp, epochs=20, batch_size=64, validation_split=0.2)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 2169554.0000 - val_loss: 1613227.8750
Epoch 2/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1444033.2500 - val_loss: 948835.6250
Epoch 3/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 908012.8750 - val_loss: 620481.8750
Epoch 4/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 696492.1875 - val_loss: 554623.0000
Epoch 5/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 646637.6875 - val_loss: 549898.7500
Epoch 6/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 633228.0625 - val_loss: 548007.8750
Epoch 7/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 625178.2500 - val_loss: 543453.9375
Epoch 8/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 617586.8750 - val_loss: 536713.6250
Epoch 9/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 609846.6250 - val_loss: 530684.9375
Epoch 10/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 602058.0625 - val_loss: 524768.3750
Epoch 11/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms

In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('kalendermenu_hpp_model.tflite', 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: /tmp/tmp24vham3v/assets


INFO:tensorflow:Assets written to: /tmp/tmp24vham3v/assets


Saved artifact at '/tmp/tmp24vham3v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 501), dtype=tf.float32, name='keras_tensor_2')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134468653525392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134468653526160: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1779972863.187070      58 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779972863.187094      58 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [18]:
bahan_rendang = "daging sapi santan cabai merah serai lengkuas bawang merah"
porsi_pesanan = 50.0  

vektor_bahan = vectorizer.transform([bahan_rendang]).toarray()

vektor_porsi = np.array([[porsi_pesanan]])

input_tes = np.hstack((vektor_bahan, vektor_porsi))

prediksi_hpp = model.predict(input_tes, verbose=0)
hasil_rupiah = int(prediksi_hpp[0][0])

print("=== 🧪 HASIL UJI COBA PREDIKSI HPP ===")
print(f"Menu        : Rendang Sapi")
print(f"Bahan Masak : {bahan_rendang}")
print(f"Total Porsi : {int(porsi_pesanan)} porsi")
print(f"--------------------------------------")
print(f"🤖 Estimasi Modal (HPP): Rp {hasil_rupiah:,}".replace(',', '.'))

=== 🧪 HASIL UJI COBA PREDIKSI HPP ===
Menu        : Rendang Sapi
Bahan Masak : daging sapi santan cabai merah serai lengkuas bawang merah
Total Porsi : 50 porsi
--------------------------------------
🤖 Estimasi Modal (HPP): Rp 1.973.548


In [19]:
import json

vocab = vectorizer.get_feature_names_out().tolist()

with open('kalendermenu_vocab.json', 'w') as f:
    json.dump(vocab, f)

print("Kamus 500 bahan berhasil diekspor ke 'kalendermenu_vocab.json'!")
print("Contoh 10 bahan pertama:", vocab[:10])

Kamus 500 bahan berhasil diekspor ke 'kalendermenu_vocab.json'!
Contoh 10 bahan pertama: ['abc', 'acar', 'ada', 'adas', 'adonan', 'agak', 'agar', 'air', 'airnya', 'aja']


In [20]:
import pandas as pd
import glob
import json

print("1. Menarik ulang data CSV dari Kaggle ke dalam memori...")
df = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)], ignore_index=True)

print(f"✅ Berhasil memuat {len(df)} baris data!")
print("2. Memproses resep menjadi Database AI untuk Wafa...")

df_clean = df[['Title', 'Ingredients']].dropna()

kamus_resep = {}

for index, row in df_clean.iterrows():
    judul = str(row['Title']).lower().strip()
    
    bahan = str(row['Ingredients']).replace('--', ', ')
    
    if judul and judul not in kamus_resep:
        kamus_resep[judul] = bahan

file_db = 'database_resep_bersih.json'
with open(file_db, 'w') as f:
    json.dump(kamus_resep, f, indent=4)

print(f"✅ Selesai! File '{file_db}' berhasil dibuat.")

1. Menarik ulang data CSV dari Kaggle ke dalam memori...
✅ Berhasil memuat 15641 baris data!
2. Memproses resep menjadi Database AI untuk Wafa...
✅ Selesai! File 'database_resep_bersih.json' berhasil dibuat.
